LSTM model 

In [16]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

NETTOYAGE + GRID TEMPOREL

In [17]:
df = pd.read_csv("data.csv")

# ===== FORMAT =====
df["WELL"] = df["WELL"].astype(str).str.strip().str.upper()
df["DATE"] = pd.to_datetime(df["DATE"])

df = df.sort_values(["WELL","DATE"])

# ===== GRID COMPLETE =====
all_wells = []

for well in df["WELL"].unique():

    df_w = df[df["WELL"] == well].copy()

    # timeline complète
    full_dates = pd.date_range(df["DATE"].min(), df["DATE"].max())

    df_w = df_w.set_index("DATE").reindex(full_dates)
    df_w.index.name = "DATE"
    df_w.reset_index(inplace=True)

    df_w["WELL"] = well

    # ===== VARIABLES PRODUCTION =====
    df_w["HOURS"] = df_w["HOURS"].fillna(0)
    df_w["W_GAS"] = df_w["W_GAS"].fillna(0)
    df_w["WATER"] = df_w["WATER"].fillna(0)
    df_w["COND_VOL"] = df_w["COND_VOL"].fillna(0)

    # ===== INTERPOLATION (limitée) =====
    df_w["WHP"] = df_w["WHP"].interpolate(limit=3)
    df_w["WHT"] = df_w["WHT"].interpolate(limit=3)

    # ===== SI ARRET =====
    df_w.loc[df_w["HOURS"] == 0, ["WHP","WHT"]] = 0

    all_wells.append(df_w)

df_clean = pd.concat(all_wells, ignore_index=True)

df_clean = df_clean.sort_values(["WELL","DATE"])
# ===== CORRECTION WATER =====

df_clean["WATER"] = df_clean["WATER"].clip(lower=0)

df_clean["WATER_LOG"] = np.log1p(df_clean["WATER"])

In [18]:
print(df_clean["WATER"].describe())

count    759033.000000
mean          1.499220
std           4.703127
min           0.000000
25%           0.000000
50%           0.065035
75%           1.479879
max         114.572266
Name: WATER, dtype: float64


FEATURES ADAPTÉES LSTM

In [19]:
# arrêt du puits
df_clean["IS_STOPPED"] = (df_clean["HOURS"] == 0).astype(int)

# vieillissement puits
df_clean["DAYS_SINCE_START"] = df_clean.groupby("WELL").cumcount()

# (optionnel mais utile)
df_clean["MONTH"] = df_clean["DATE"].dt.month

In [20]:
features = [
    "WHP",
    "WHT",
    "HOURS",
    "IS_STOPPED",
    "DAYS_SINCE_START",
    "MONTH"
]

targets = ["W_GAS","WATER_LOG","COND_VOL"]

SPLIT PAR WELL

In [21]:
wellnames = df_clean["WELL"].unique().tolist()

np.random.seed(42)
np.random.shuffle(wellnames)

train_wells = wellnames[:int(0.8*len(wellnames))]
blind_wells = wellnames[int(0.8*len(wellnames)):]

CREATE SEQUENCES (LSTM)

In [22]:
def create_sequences(df, features, targets, window=7):

    X_list = []
    y_list = []

    for well in df["WELL"].unique():

        df_w = df[df["WELL"] == well].sort_values("DATE")

        X_w = df_w[features].values
        y_w = df_w[targets].values

        for i in range(window, len(df_w)):

            X_list.append(X_w[i-window:i])
            y_list.append(y_w[i])

    return np.array(X_list), np.array(y_list)

DATA LSTM

In [23]:
window = 7

# TRAIN
df_train = df_clean[df_clean["WELL"].isin(train_wells)]
X_seq, y_seq = create_sequences(df_train, features, targets, window)

# SPLIT interne
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=1
)

# BLIND
df_blind = df_clean[df_clean["WELL"].isin(blind_wells)]
X_blind, y_blind = create_sequences(df_blind, features, targets, window)

NORMALISATION 

In [24]:
scaler = StandardScaler()

nsamples, ntimestep, nfeatures = X_train.shape

X_train_reshaped = X_train.reshape(-1, nfeatures)
X_test_reshaped  = X_test.reshape(-1, nfeatures)
X_blind_reshaped = X_blind.reshape(-1, nfeatures)

X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_test_scaled  = scaler.transform(X_test_reshaped)
X_blind_scaled = scaler.transform(X_blind_reshaped)

X_train_scaled = X_train_scaled.reshape(X_train.shape)
X_test_scaled  = X_test_scaled.reshape(X_test.shape)
X_blind_scaled = X_blind_scaled.reshape(X_blind.shape)

MODEL LSTM (OPTIMISÉ)

In [25]:
model = Sequential([

    LSTM(64, return_sequences=True, input_shape=(window, len(features))),
    Dropout(0.2),

    LSTM(32),
    Dropout(0.2),

    Dense(32, activation='relu'),
    Dense(3)
])

model.compile(
    optimizer='adam',
    loss='mse'
)

model.summary()

a:\pfe master\PEG_Python-master\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 7, 64)          │        18,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 7, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,747 (124.01 KB)

 Trainable params: 31,747 (124.01 KB)

 Non-trainable params: 0 (0.00 B)

TRAINING

In [26]:
model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=25,
    batch_size=256,
    verbose=1
)

Epoch 1/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 59s 28ms/step - loss: 137.0025 - val_loss: 87.7223
Epoch 2/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 43s 23ms/step - loss: 87.7377 - val_loss: 80.4717
Epoch 3/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 45s 24ms/step - loss: 83.0277 - val_loss: 78.6788
Epoch 4/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 44s 23ms/step - loss: 80.1823 - val_loss: 77.0393
Epoch 5/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 38s 20ms/step - loss: 78.4598 - val_loss: 75.3850
Epoch 6/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 43s 23ms/step - loss: 77.5758 - val_loss: 73.3163
Epoch 7/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 100s 32ms/step - loss: 75.8911 - val_loss: 73.8538
Epoch 8/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 66s 24ms/step - loss: 74.7944 - val_loss: 74.4357
Epoch 9/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 82s 23ms/step - loss: 73.8219 - val_loss: 71.1077
Epoch 10/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 44s 23ms/step - loss: 72.9182 - val_loss: 70.5951
Epoch 11/25
1892/1892 ━━━━━━━━━━━━━━━━━━━━ 42s 22ms/step - loss: 71.8738 - va

EVALUATION

In [27]:
y_pred = model.predict(X_blind_scaled)
# ===== RECONVERTIR WATER =====

y_pred_real = y_pred.copy()
y_blind_real = y_blind.copy()

# index 1 = WATER_LOG
y_pred_real[:,1] = np.expm1(y_pred[:,1])
y_blind_real[:,1] = np.expm1(y_blind[:,1])
for i, target in enumerate(["W_GAS","WATER","COND_VOL"]):
    print(target)
    print("R2:", r2_score(y_blind_real[:, i], y_pred_real[:, i]))

4788/4788 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step
W_GAS
R2: 0.7490285089056585
WATER
R2: 0.06111632704020675
COND_VOL
R2: 0.7368825077783783


🔥 12. BONUS : Détection anomalies (simple)

In [28]:
errors = np.abs(y_blind - y_pred)

threshold = errors.mean() + 2 * errors.std()

anomalies = errors > threshold

print("Anomalies détectées:", anomalies.sum())

Anomalies détectées: 17630


analysis

In [30]:
print("===== NaN PAR COLONNE =====")
print(df_clean.isna().sum())

print("\n===== % NaN =====")
print((df_clean.isna().mean() * 100))

===== NaN PAR COLONNE =====
DATE                     0
WELL                     0
HOURS                    0
WHP                      0
WHT                      0
C2M                 222815
C3                  222815
C4                  222815
C5P                 222815
H2O                 222815
W_GAS                    0
S_GAS               222815
LPG_VOL             222815
LPG_MASS            222815
COND_VOL                 0
COND_MASS           222815
WATER                    0
prodindex           759032
WLP                 396658
WATER_LOG                0
IS_STOPPED               0
DAYS_SINCE_START         0
MONTH                    0
dtype: int64

===== % NaN =====
DATE                 0.000000
WELL                 0.000000
HOURS                0.000000
WHP                  0.000000
WHT                  0.000000
C2M                 29.355114
C3                  29.355114
C4                  29.355114
C5P                 29.355114
H2O                 29.355114
W_GAS              

In [31]:
print("\n===== NOMBRE DE ZEROS PAR COLONNE =====")

for col in ["WHP","WHT","W_GAS","WATER","COND_VOL","HOURS"]:
    zero_count = (df_clean[col] == 0).sum()
    total = len(df_clean)
    print(f"{col}: {zero_count} ({100*zero_count/total:.2f}%)")


===== NOMBRE DE ZEROS PAR COLONNE =====
WHP: 273326 (36.01%)
WHT: 276565 (36.44%)
W_GAS: 274727 (36.19%)
WATER: 348866 (45.96%)
COND_VOL: 274727 (36.19%)
HOURS: 273259 (36.00%)


In [32]:
summary = pd.DataFrame({
    "NaN": df_clean.isna().sum(),
    "Zero": (df_clean == 0).sum(),
    "Total": len(df_clean)
})

summary["% NaN"] = summary["NaN"] / summary["Total"] * 100
summary["% Zero"] = summary["Zero"] / summary["Total"] * 100

print(summary)

                     NaN    Zero   Total      % NaN     % Zero
DATE                   0       0  759033   0.000000   0.000000
WELL                   0       0  759033   0.000000   0.000000
HOURS                  0  273259  759033   0.000000  36.000938
WHP                    0  273326  759033   0.000000  36.009765
WHT                    0  276565  759033   0.000000  36.436492
C2M               222815   51912  759033  29.355114   6.839228
C3                222815   51912  759033  29.355114   6.839228
C4                222815   51912  759033  29.355114   6.839228
C5P               222815   51912  759033  29.355114   6.839228
H2O               222815  126130  759033  29.355114  16.617196
W_GAS                  0  274727  759033   0.000000  36.194342
S_GAS             222815   51912  759033  29.355114   6.839228
LPG_VOL           222815   51912  759033  29.355114   6.839228
LPG_MASS          222815   51912  759033  29.355114   6.839228
COND_VOL               0  274727  759033   0.000000  36

In [33]:
print("\n===== DESCRIPTION NUMERIQUE =====")
print(df_clean.describe())

print("\n===== TYPES DE DONNEES =====")
print(df_clean.dtypes)

print("\n===== NOMBRE DE PUITS =====")
print(df_clean["WELL"].nunique())

print("\n===== NOMBRE DE LIGNES =====")
print(len(df_clean))

print("\n===== PERIODE =====")
print(df_clean["DATE"].min(), "→", df_clean["DATE"].max())


===== DESCRIPTION NUMERIQUE =====
                      DATE          HOURS            WHP            WHT  \
count               759033  759033.000000  759033.000000  759033.000000   
mean   2009-09-01 00:00:00      14.864520      44.257418      36.622689   
min    1999-03-05 00:00:00       0.000000       0.000000       0.000000   
25%    2004-06-02 00:00:00       0.000000       0.000000       0.000000   
50%    2009-09-01 00:00:00      24.000000      43.000000      52.000000   
75%    2014-12-01 00:00:00      24.000000      80.000000      61.000000   
max    2020-02-29 00:00:00      25.000000     694.000000     587.000000   
std                    NaN      11.442399      38.405691      28.376816   

                 C2M             C3             C4            C5P  \
count  536218.000000  536218.000000  536218.000000  536218.000000   
mean        0.222277       0.013044       0.005989       0.006989   
min        -0.055233      -0.003223      -0.001396      -0.001290   
25%         0

In [34]:
print("\n===== WATER =====")
print(df_clean["WATER"].describe())

print("\n===== WATER_LOG =====")
print(df_clean["WATER_LOG"].describe())


===== WATER =====
count    759033.000000
mean          1.499220
std           4.703127
min           0.000000
25%           0.000000
50%           0.065035
75%           1.479879
max         114.572266
Name: WATER, dtype: float64

===== WATER_LOG =====
count    759033.000000
mean          0.507640
std           0.711919
min           0.000000
25%           0.000000
50%           0.063007
75%           0.908210
max           4.749896
Name: WATER_LOG, dtype: float64
